# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hanizakkk/flyrank_working-repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** flag a content item for refresh review if it
already earns real search visibility (impressions at or above the population
median — 246 impressions in March 2026) but is wasting that visibility —
either sitting off page one on average (`avg_position >= 10`) or converting
visibility into clicks worse than a typical visible item (`ctr < 0.0017`, the
median CTR among visible items). The score is sized by raw impressions, so a
high-traffic underperformer ranks above a low-traffic one with the same flags.

**Reason codes:** `visible_weak_position`, `visible_low_ctr` (an item can carry
both), `general_refresh_review` (visible, neither specific flag tripped),
`not_visible_enough` (below the visibility floor — monitor only, not scored).

Run in Colab: `capstone_full_pipeline.ipynb`, section 3. Results below are
copied from that run (March 2026 decision window -> April 2026 outcome
window, 158,549 items).

In [1]:
# Full run happened in work/notebooks/capstone_full_pipeline.ipynb (section 3),
# against the real warehouse via Colab. This cell mirrors that logic for the
# record; the numbers reported below are copied from that actual run's
# baseline_metrics.json and ranked_recommendations_sample.json, not re-simulated here.

import json
import pandas as pd

with open("../outputs/baseline_metrics.json") as f:
    baseline_metrics = json.load(f)

print(json.dumps(baseline_metrics, indent=2))
# Expected (from the real run):
#   decision_month=2026-03, outcome_month=2026-04, n_rows=158549
#   base_rate=0.591, precision_at_50=0.76, precision_at_100=0.78
#   thresholds: impressions_visible=246.0, position_weak=10.0, ctr_low=0.00166

{
  "base_rate": 0.5914827592731584,
  "decision_month": "2026-03",
  "n_rows": 158549,
  "outcome_month": "2026-04",
  "precision_at_100": 0.78,
  "precision_at_50": 0.76,
  "thresholds": {
    "ctr_low": 0.0016611295681063123,
    "impressions_visible": 246.0,
    "position_weak": 10.0
  }
}


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
ranked = pd.read_json("../outputs/ranked_recommendations_sample.json")
# Note: this CSV is from the June 2026 LIVE scoring pass (ML-10), which reuses
# the same rule/reason-code logic. The March->April DEV ranking used for the
# precision numbers above lives in the pipeline notebook's in-memory
# `ranked_baseline` (not separately exported, since only aggregated metrics -
# not full per-item dev rankings - need to be committed).
ranked.head(20)

,rank,client_hash_id,content_hash_id,model_score,action,reason_code,confidence,impressions,clicks,avg_position,ctr
0,1,client_b10cb2997d0c7c86,content_87eb4619444da89f,0.833023,refresh_priority,weak_position|low_ctr,high,646,0,28.099554,0.000000
1,2,client_810019792c9b8efc,content_826a6d837682df3c,0.833020,refresh_priority,weak_position|low_ctr,high,498,0,10.845758,0.000000
2,3,client_9958f0a7ae1df715,content_572263a160b6c8ab,0.830888,refresh_priority,weak_position|low_ctr,high,510,0,26.276404,0.000000
3,4,client_73cda7b4e4f265ea,content_1f10cce14fa14a4c,0.830485,refresh_priority,low_ctr,high,338,0,9.095340,0.000000
4,5,client_9958f0a7ae1df715,content_d8288f22519f7d53,0.830259,refresh_priority,weak_position|low_ctr,high,633,0,38.063023,0.000000
5,6,client_b10cb2997d0c7c86,content_5d2c0a2568ca0bac,0.828595,refresh_priority,low_ctr,high,1315,0,8.921163,0.000000
6,7,client_73cda7b4e4f265ea,content_ebe1f0da038a3a54,0.826721,refresh_priority,weak_position|low_ctr,high,2283,2,15.898636,0.000876
7,8,client_b10cb2997d0c7c86,content_8b1d28df3ed4e324,0.825458,refresh_priority,weak_position|low_ctr,high,1176,1,38.878251,0.000850
8,9,client_23a62021009f63c4,content_11a15d9c9ff6f9ea,0.824141,refresh_priority,weak_position|low_ctr,high,651,0,30.984887,0.000000
9,10,client_fef1a8f436438636,content_3aba7dc3b43d583f,0.823872,refresh_priority,weak_position|low_ctr,high,926,0,21.630945,0.000000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top-20 review (from the real March->April dev ranking):** the highest-scoring
items were consistently high-impression, high-position (page 2+) items with
near-zero CTR — exactly the pattern the rule is designed to surface. Precision@50
of 0.76 against a 59.1% base rate means the rule beats random guessing by roughly
+17 points at the top of the queue, but is far from perfect: about 1 in 4 of the
top 50 flagged items did not actually decline the following month.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** items with `not_visible_enough` never enter the ranked queue at
all (score=0 by construction) - the biggest limitation of this rule is that it
says nothing about low-traffic content, which is exactly where "worth writing
from scratch vs. not worth touching" judgment calls are hardest.

**Leakage check:** all four features (`impressions`, `avg_position`, `ctr`, plus
the four not used in the rule itself) are aggregated over March 2026 only; the
label comes from April 2026 only via a separate query, and the two are never
computed from the same SQL statement (see `aggregate_month_features` vs.
`aggregate_month_outcome` in `work/scripts/warehouse_utils.py` / the pipeline
notebook). `client_hash_id`/`content_hash_id` are used only to join and group,
never passed to the scoring rule.

In [4]:
# Leakage checklist, copied from work/outputs/leakage_audit.json (real run):
import json
with open("../outputs/leakage_audit.json") as f:
    audit = json.load(f)
print(json.dumps(audit, indent=2))
# All four checks PASS. The deliberate leak test (label folded into its own
# features) hit precision@50 = 1.000 - confirming the audit harness correctly
# detects leakage when it is deliberately introduced.

{
  "checklist": {
    "features_only_from_decision_month": true,
    "grouped_split_used_for_reported_results": true,
    "ids_excluded_from_features": true,
    "no_label_or_sibling_columns_in_features": true
  },
  "leaky_precision_at_50": 1.0
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.